In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import joblib

feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

# from FEATURE_ENGINEERING.features import *
from MODELS.model import *
from MODELS.pipeline import *
pd.set_option('display.max_columns', None)

c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\venv310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Star Players
    # 'PLAYER_IS_TEAM_STAR', 'TEAM_STAR_OUT',
    # 'PTS_WITHOUT_STAR', 'MIN_WITHOUT_STAR', 'USG_PCT_WITHOUT_STAR', 'FGA_WITHOUT_STAR', 'FG3A_WITHOUT_STAR', 'FTA_WITHOUT_STAR', 
    # 'EFG_PCT_WITHOUT_STAR', 'TS_PCT_WITHOUT_STAR', 'AST_WITHOUT_STAR', 'REB_WITHOUT_STAR', 'PTS_PER_36_WITHOUT_STAR',
    
    # Player season averages
    'MIN_SEASON_AVG_TO_DATE', 'PTS_SEASON_AVG_TO_DATE','FGA_SEASON_AVG_TO_DATE','FG3A_SEASON_AVG_TO_DATE',
    'FTA_SEASON_AVG_TO_DATE','USG_PCT_SEASON_AVG_TO_DATE','TS_PCT_SEASON_AVG_TO_DATE',
    'EFG_PCT_SEASON_AVG_TO_DATE', 'AST_SEASON_AVG_TO_DATE', 'REB_SEASON_AVG_TO_DATE', 'TOV_SEASON_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2',
    'FGA_LAG_1', 'FGA_LAG_2',
    'MIN_LAG_1', 'MIN_LAG_2',
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2',
    
    # Short-term form (5-game rolling averages)
    'MIN_ROLLING_AVG_5', 'PTS_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5',
    'FG3A_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5',
    'TS_PCT_ROLLING_AVG_5','EFG_PCT_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 
    'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # Medium-term form (15-game rolling averages)
    'MIN_ROLLING_AVG_15', 'PTS_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15',
    'FG3A_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15',
    'TS_PCT_ROLLING_AVG_15','EFG_PCT_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 
    'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'MIN_ROLLING_AVG_40', 'PTS_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40',
    'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 
    'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',

    # Matchup micro-feature
    'MATCHUP_AVG_MIN_LAST_3_TO_DATE', 'MATCHUP_AVG_FGA_LAST_3_TO_DATE', 'MATCHUP_AVG_FG3A_LAST_3_TO_DATE', 'MATCHUP_AVG_FTA_LAST_3_TO_DATE', 
    'MATCHUP_AVG_PTS_LAST_3_TO_DATE', 'MATCHUP_AVG_USG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_EFG_PCT_LAST_3_TO_DATE', 'MATCHUP_AVG_TS_PCT_LAST_3_TO_DATE', 
    'MATCHUP_AVG_AST_LAST_3_TO_DATE', 'MATCHUP_AVG_REB_LAST_3_TO_DATE', 'MATCHUP_AVG_TOV_LAST_3_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]


In [20]:
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
odds = pd.read_csv('../BACKTESTS/odds25.csv')
bookmakers = odds[(odds['GAME_DATE'] == '2025-04-13') & (odds['BOOKMAKER'] == 'prizepicks') & (odds['CATEGORY'] =='points')]
model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
# model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
starters = getStarters(22400307, 'CLE', data)
games = get_espn_games(20250413)

C:\Users\alexg\AppData\Local\Temp\ipykernel_23908\2910744972.py:2: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  odds = pd.read_csv('../BACKTESTS/odds25.csv')


In [22]:
def makePredictionCatBoost(player_name, data, model, bookmakers, games, todayDate, starters, game_id, features, n_games=3):
    from catboost import Pool
    import pandas as pd
    
    # Get feature vector
    feature_vector = buildFeatureVector(player_name, data, games, todayDate, starters, game_id, n_games=n_games)
    
    # Convert to DataFrame with proper feature names
    X = pd.DataFrame([feature_vector], columns=features)
    
    # Define categorical features (same as used during training)
    categorical_cols = ['PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID']
    cat_cols = [c for c in categorical_cols if c in features]
    cat_idx = [features.index(c) for c in cat_cols]
    
    # Data type cleanup (matching training preprocessing)
    for c in X.columns:
        if c not in cat_cols:
            if X[c].dtype == 'bool':
                X[c] = X[c].astype(int)
            elif X[c].dtype == 'object':
                X[c] = pd.to_numeric(X[c], errors='coerce')
    
    # Create CatBoost Pool with categorical features
    pool = Pool(X, cat_features=cat_idx)
    
    # Make prediction
    prediction = model.predict(pool)[0]
    
    prop_line = bookmakers[bookmakers['NAME'] == player_name]['LINE'].values[0]
    
    # Get opponent team for display
    player_id, player_team = findPlayerID(player_name, data)
    opponent, homeGame = findOppTeam(player_name, data, games)
    
    return {
        'player': player_name,
        'opponent': opponent,
        'predicted_stat': round(prediction, 2),
        'raw_prediction': prediction,
        'prop_line': prop_line,
        'edge': round(prediction - prop_line, 2),
        'recommendation': 'OVER' if prediction > prop_line else 'UNDER'
    }

pred = makePredictionCatBoost('LeBron James', data, model, bookmakers, games, 20250413, starters, game_id=22401185, features=features)
pred

{'player': 'LeBron James',
 'opponent': 'POR',
 'predicted_stat': np.float64(19.85),
 'raw_prediction': np.float64(19.852101508212144),
 'prop_line': np.float64(24.5),
 'edge': np.float64(-4.65),
 'recommendation': 'UNDER'}

In [55]:
import requests
import pandas as pd
import json
from typing import List, Optional
import os

API_BASE = "https://api.sportsgameodds.com/v2"
API_KEY = '86342e4cecd1063f9977239707a88da3'
HEADERS = {"X-Api-Key": API_KEY}

# Only allow these stat types
STAT_ALLOW = {"PTS", "AST", "REB", "points", "assists", "rebounds"}

def build_player_index(players) -> dict:
    """
    Returns a dict mapping every known key to the player object
    """
    index = {}

    if not players:
        return index

    if isinstance(players, dict) and "byID" in players and isinstance(players["byID"], dict):
        iterable = players["byID"].values()
    elif isinstance(players, dict):
        iterable = players.values()
    elif isinstance(players, list):
        iterable = players
    else:
        iterable = []

    for p in iterable:
        if not isinstance(p, dict):
            continue
        pid = p.get("playerID") or p.get("id") or p.get("entityID")
        if pid:
            index[pid] = p
        for alias in p.get("aliases") or []:
            if isinstance(alias, str):
                index[alias] = p

    return index

def resolve_player(players_index: dict, candidate_id: str) -> dict:
    if not candidate_id:
        return {}
    return players_index.get(candidate_id, {})

def collect_nba_odds_formatted_fixed(
    starts_after: str,
    starts_before: str,
    save_to_csv: bool = True,
    output_file: Optional[str] = None
) -> pd.DataFrame:
    print(f"📊 Collecting NBA Player Props - FIXED VERSION")
    print(f"📅 {starts_after} to {starts_before}")
    print("=" * 55)

    all_event_data = []
    next_cursor = None
    page_count = 0

    while True:
        page_count += 1
        print(f"📡 Fetching page {page_count}...")

        params = {
            "leagueIDs": "NBA",  # Fixed: was "leagueID"
            "startsAfter": starts_after,
            "startsBefore": starts_before,
            "oddsPresent": "true",
            "includeAltLines": "true",
            "limit": 100
        }
        if next_cursor:
            params["cursor"] = next_cursor

        r = requests.get(f"{API_BASE}/events", headers=HEADERS, params=params, timeout=30)
        if r.status_code != 200:
            print(f"❌ API error: {r.status_code} - {r.text}")
            break

        data = r.json()
        events = data.get("data", [])
        if not events:
            break

        print(f"   ✅ Found {len(events)} events")
        all_event_data.extend(events)
        next_cursor = data.get("nextCursor")
        if not next_cursor:
            break

    print(f"✅ Total events: {len(all_event_data)}")
    if not all_event_data:
        return pd.DataFrame()

    formatted_records = []
    players_found = set()
    stats_found = {}

    for event in all_event_data:
        starts_at = event.get("startsAt", "")
        game_date = starts_at[:10] if starts_at else ""
        event_id = event.get("eventID", "")
        home_team = event.get("homeTeam", "")
        away_team = event.get("awayTeam", "")
        matchup = f"{away_team} @ {home_team}" if away_team and home_team else ""

        players_raw = event.get("players") or {}
        players_index = build_player_index(players_raw)

        odds = event.get("odds") or {}
        for odd_id, odd_object in odds.items():
            # Player identifier
            stat_entity_id = (
                odd_object.get("statEntityID")
                or odd_object.get("entityID")
            )
            if not stat_entity_id:
                parts = str(odd_id).split("-")
                if len(parts) >= 2:
                    stat_entity_id = parts[1]

            # Stat type
            stat_id = odd_object.get("statID")
            if not stat_id:
                parts = str(odd_id).split("-")
                stat_id = parts[0] if parts else None

            bet_type = odd_object.get("betTypeID")
            if not bet_type:
                parts = str(odd_id).split("-")
                bet_type = parts[3] if len(parts) > 3 else None

            # FIXED: Filter for over/under bets instead of "prop"
            if bet_type != "ou":  # Changed from "prop" to "ou"
                continue
            if stat_id not in STAT_ALLOW:
                continue
            if stat_entity_id in {"home", "away", "all", None}:
                continue

            # Track what we're finding
            stats_found[stat_id] = stats_found.get(stat_id, 0) + 1

            player_info = resolve_player(players_index, stat_entity_id)
            names = player_info.get("names") or {}
            player_name = names.get("display") or names.get("full") or player_info.get("name") or ""
            
            # If no player name from player info, try to extract from market name
            if not player_name:
                market_name = odd_object.get("marketName", "")
                if market_name and stat_id:
                    # Extract from "Player Name Points Over/Under"
                    import re
                    pattern = rf'(.+?)\s+{stat_id.title()}\s+Over/Under'
                    match = re.search(pattern, market_name, re.IGNORECASE)
                    if match:
                        player_name = match.group(1).strip()
            
            if player_name:
                players_found.add(player_name)
            
            team_id = player_info.get("teamID", "")
            side_id = odd_object.get("sideID", "")
            period_id = odd_object.get("periodID", "")
            market_name = odd_object.get("marketName", "")
            actual_score = odd_object.get("score")

            fair_line = odd_object.get("closeFairOverUnder") or odd_object.get("fairOverUnder")
            fair_odds = odd_object.get("closeFairOdds") or odd_object.get("fairOdds")
            book_line_cons = odd_object.get("closeBookOverUnder") or odd_object.get("bookOverUnder")
            book_odds_cons = odd_object.get("closeBookOdds") or odd_object.get("bookOdds")

            by_bookmaker = odd_object.get("byBookmaker") or {}
            if isinstance(by_bookmaker, str):
                try:
                    by_bookmaker = json.loads(by_bookmaker)
                except Exception:
                    by_bookmaker = {}

            def make_record(bookmaker_name, bookmaker_data):
                return {
                    "game_date": game_date,
                    "event_id": event_id,
                    "home_team": home_team,
                    "away_team": away_team,
                    "matchup": matchup,
                    "player_id": stat_entity_id,
                    "player_name": player_name,
                    "team_id": team_id,
                    "stat_type": stat_id,
                    "period_id": period_id,
                    "bet_type": bet_type,
                    "side_id": side_id,
                    "market_name": market_name,
                    "bookmaker": bookmaker_name,
                    "book_line": bookmaker_data.get("overUnder") or bookmaker_data.get("spread"),
                    "book_odds": bookmaker_data.get("odds"),
                    "fair_line": fair_line,
                    "fair_odds": fair_odds,
                    "available": bookmaker_data.get("available", True),
                    "actual_score": actual_score,
                    "odd_id": odd_id,
                    "started": odd_object.get("started", False),
                    "ended": odd_object.get("ended", False),
                    "last_updated": bookmaker_data.get("lastUpdatedAt", "")
                }

            if isinstance(by_bookmaker, dict) and by_bookmaker:
                for bookmaker_name, bookmaker_data in by_bookmaker.items():
                    if isinstance(bookmaker_data, dict):
                        formatted_records.append(make_record(bookmaker_name, bookmaker_data))
            else:
                formatted_records.append({
                    "game_date": game_date,
                    "event_id": event_id,
                    "home_team": home_team,
                    "away_team": away_team,
                    "matchup": matchup,
                    "player_id": stat_entity_id,
                    "player_name": player_name,
                    "team_id": team_id,
                    "stat_type": stat_id,
                    "period_id": period_id,
                    "bet_type": bet_type,
                    "side_id": side_id,
                    "market_name": market_name,
                    "bookmaker": "consensus",
                    "book_line": book_line_cons,
                    "book_odds": book_odds_cons,
                    "fair_line": fair_line,
                    "fair_odds": fair_odds,
                    "available": True,
                    "actual_score": actual_score,
                    "odd_id": odd_id,
                    "started": odd_object.get("started", False),
                    "ended": odd_object.get("ended", False),
                    "last_updated": ""
                })

    if not formatted_records:
        print("❌ No formatted records created")
        return pd.DataFrame()

    df = pd.DataFrame(formatted_records)
    print(f"📊 Created {len(df)} formatted records")
    
    # Show summary
    print(f"\n👥 Found {len(players_found)} unique players:")
    for name in sorted(list(players_found)[:10]):
        print(f"   • {name}")
    if len(players_found) > 10:
        print(f"   ... and {len(players_found) - 10} more")
    
    print(f"\n📊 Stats breakdown: {stats_found}")
    
    if 'bookmaker' in df.columns:
        bookmaker_counts = df['bookmaker'].value_counts()
        print(f"\n📚 Top bookmakers:")
        for bookmaker, count in bookmaker_counts.head(5).items():
            print(f"   {bookmaker}: {count}")

    if save_to_csv:
        if output_file is None:
            start_clean = starts_after.replace("-", "")
            end_clean = starts_before.replace("-", "")
            output_file = f"data2.csv"
        os.makedirs("BACKTESTS", exist_ok=True)
        path = os.path.join("BACKTESTS", output_file)
        df.to_csv(path, index=False)
        print(f"💾 Saved to: {path}")

    return df
    
if __name__ == "__main__":
    df = collect_nba_odds_formatted_fixed(
        starts_after="2025-01-01",
        starts_before="2025-04-12"
    )
    
    if not df.empty:
        print(f"\n📋 Sample data:")
        sample_cols = ['player_name', 'stat_type', 'side_id', 'bookmaker', 'book_line', 'actual_score']
        available_cols = [col for col in sample_cols if col in df.columns]
        print(df[available_cols].head(10))

📊 Collecting NBA Player Props - FIXED VERSION
📅 2025-01-01 to 2025-04-12
📡 Fetching page 1...
   ✅ Found 25 events
📡 Fetching page 2...
   ✅ Found 25 events
📡 Fetching page 3...
   ✅ Found 25 events
📡 Fetching page 4...
   ✅ Found 25 events
📡 Fetching page 5...
   ✅ Found 25 events
📡 Fetching page 6...
   ✅ Found 25 events
📡 Fetching page 7...
   ✅ Found 25 events
📡 Fetching page 8...
   ✅ Found 25 events
📡 Fetching page 9...
   ✅ Found 25 events
📡 Fetching page 10...
   ✅ Found 25 events
📡 Fetching page 11...
   ✅ Found 25 events
📡 Fetching page 12...
   ✅ Found 25 events
📡 Fetching page 13...
   ✅ Found 25 events
📡 Fetching page 14...
   ✅ Found 25 events
📡 Fetching page 15...
   ✅ Found 25 events
📡 Fetching page 16...
   ✅ Found 25 events
📡 Fetching page 17...
   ✅ Found 25 events
📡 Fetching page 18...
   ✅ Found 25 events
📡 Fetching page 19...
   ✅ Found 25 events
📡 Fetching page 20...
   ✅ Found 25 events
📡 Fetching page 21...
   ✅ Found 25 events
📡 Fetching page 22...
   ✅ Found 

In [74]:
from datetime import datetime
from zoneinfo import ZoneInfo

def convert_to_et(utc_time):
    utc_dt = datetime.fromisoformat(utc_time.replace('Z', '+00:00'))
    et_dt = utc_dt.astimezone(ZoneInfo("America/New_York"))
    return et_dt.strftime('%Y-%m-%d')

df = pd.read_csv('BACKTESTS/data2.csv')
data = df[['player_name', 'stat_type', 'side_id', 'bookmaker', 'book_line', 'book_odds', 'last_updated']]
data.rename(columns={'player_name': 'NAME', 'stat_type': 'CATEGORY', 'side_id': 'SIDE', 'bookmaker': 'BOOKMAKER', 'book_line': 'LINE', 'book_odds': 'ODDS', 'last_updated': 'GAME_DATE'}, inplace=True)

data['GAME_DATE'] = pd.to_datetime(data['GAME_DATE']).dt.tz_convert('America/New_York').dt.strftime('%Y-%m-%d')
data.to_csv('BACKTESTS/newData2.csv', index=False)

C:\Users\alexg\AppData\Local\Temp\ipykernel_10016\2868675205.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.rename(columns={'player_name': 'NAME', 'stat_type': 'CATEGORY', 'side_id': 'SIDE', 'bookmaker': 'BOOKMAKER', 'book_line': 'LINE', 'book_odds': 'ODDS', 'last_updated': 'GAME_DATE'}, inplace=True)
C:\Users\alexg\AppData\Local\Temp\ipykernel_10016\2868675205.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['GAME_DATE'] = pd.to_datetime(data['GAME_DATE']).dt.tz_convert('America/New_York').dt.strftime('%Y-%m-%d')


In [76]:
first = pd.read_csv('BACKTESTS/newData1.csv')
second = pd.read_csv('BACKTESTS/newData2.csv')
allData = pd.concat([first, second])
allData

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,GAME_DATE
0,Al Horford,rebounds,under,betparx,6.5,-162.0,2024-10-22
1,Payton Pritchard,points,over,bovada,8.5,-111.0,2024-10-22
2,Payton Pritchard,points,over,betonline,8.5,-111.0,2024-10-22
3,Josh Hart,points,under,bovada,8.5,-116.0,2024-10-22
4,Josh Hart,points,under,betonline,8.5,-116.0,2024-10-22
...,...,...,...,...,...,...,...
842969,Ty Jerome,points,over,draftkings,15.5,-110.0,2025-04-11
842970,Ty Jerome,points,under,draftkings,15.5,-120.0,2025-04-11
842971,Ty Jerome,rebounds,under,draftkings,2.5,-154.0,2025-04-11
842972,Ty Jerome,rebounds,over,draftkings,2.5,120.0,2025-04-11


In [79]:
allData[(allData['CATEGORY'] == 'points') & (allData['BOOKMAKER'] != 'unknown') & (allData['BOOKMAKER'] != 'prizepicks')]

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,GAME_DATE
1,Payton Pritchard,points,over,bovada,8.5,-111.0,2024-10-22
2,Payton Pritchard,points,over,betonline,8.5,-111.0,2024-10-22
3,Josh Hart,points,under,bovada,8.5,-116.0,2024-10-22
4,Josh Hart,points,under,betonline,8.5,-116.0,2024-10-22
5,Josh Hart,points,under,betparx,8.5,-109.0,2024-10-22
...,...,...,...,...,...,...,...
842935,Landry Shamet,points,under,espnbet,7.5,-120.0,2025-04-11
842936,Landry Shamet,points,under,underdog,7.5,100.0,2025-04-11
842937,Landry Shamet,points,under,prophetexchange,7.5,-125.0,2025-04-11
842969,Ty Jerome,points,over,draftkings,15.5,-110.0,2025-04-11


In [80]:
allData.to_csv('BACKTESTS/singleBets.csv', index=False)